## Han Solo's Transportation Problem 

Han Solo has three big crime lords he’s trying to keep happy and is planning three new smuggling routes. Each crime lord requires specific payment and each smuggling route provides certain numbers of credits. It costs Han 4 credits/parsec to travel the smuggling routes. Distances to each crime lord are given below:

| Route  | Jabba | Pyke    | Sollima | Capacity | 
|-------:|------:|--------:|--------:|---------:|
|      1 | 50    |   25    |  84     | 24       | 
|      2 | 12    |   75    |  18     | 12       |
|      3 | 64    |   39    |  70     | 9        | 
|Cred Req| 16    |   20    |  9      | = 45     |

How should Han get credits to the crime lords while minimize his costs and meeting all demands?

In [9]:
using JuMP, Clp, NamedArrays

sites = [ 1,  2,  3]
crimelords = [:J, :P, :S]


cost_per_haul = 2*4    # don't forget the return trip!

dist = NamedArray( [50 25 84; 12 75 18;64 39 70], (sites,crimelords), ("Sites","Crimelords") )
supply = Dict(zip( sites, [24 12 9] ))
demand = Dict(zip( crimelords, [16 20 9] ))

m = Model(solver=ClpSolver())

@variable(m, x[sites,crimelords] >= 0) # x[i,j] is credits obtained on route i for crimelord j

@constraint(m, sup[i in sites], sum(x[i,j] for j in crimelords) == supply[i] )   # supply constraint
@constraint(m, dem[j in crimelords], sum(x[i,j] for i in sites) == demand[j] )   # demand constraint


@objective(m, Min, cost_per_haul*sum( x[i,j]*dist[i,j] for i in sites, j in crimelords ) ) # minimize transportation cost

status = solve(m)

println(status)

# nicely formatted solution
solution = NamedArray( Int[getvalue(x[i,j]) for i in sites, j in crimelords], (sites,crimelords), ("Sites","Crimelords") )
println( solution )
println()
println("Total cost will be \$", getobjectivevalue(m))



Optimal
3×3 Named Array{Int64,2}
Sites ╲ Crimelords │ :J  :P  :S
───────────────────┼───────────
1                  │  4  20   0
2                  │  3   0   9
3                  │  9   0   0

Total cost will be $11792.0
